E01: train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model?

In [1]:
import torch
import torch.nn.functional as F

In [2]:
words = open('names.txt', 'r').read().splitlines()

In [3]:
#create mapping from strings to ints and ints to strings
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

In [4]:
#create training set
xs, ys = [], []

for w in words:
    chs = ['.', '.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]

        xs.append((ix1, ix2))
        ys.append(ix3)

xs = torch.tensor(xs)
ys= torch.tensor(ys)
num = xs[:,0].nelement()
num

228146

In [5]:
#encode x
xenc = F.one_hot(xs, num_classes=27).float()
xenc = torch.cat((xenc[:,0],xenc[:,1]), dim=1)

In [6]:
#initialize weights randomly, 54 input by 27 output
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((54, 27), requires_grad=True, generator=g)

In [7]:
for k in range(100):
    #forward pass
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(dim=1, keepdim=True)
    loss = -probs[torch.arange(num), ys].log().mean()
    print(loss.item())

    #backward pass
    W.grad = None
    loss.backward()

    #update
    W.data += -10 * W.grad

4.232540607452393
3.9882781505584717
3.8091485500335693
3.671083450317383
3.5583372116088867
3.462064027786255
3.378868579864502
3.306776762008667
3.2441036701202393
3.1893317699432373
3.1411657333374023
3.09853458404541
3.0605692863464355
3.02657151222229
2.9959771633148193
2.9683234691619873
2.943222761154175
2.920346975326538
2.8994147777557373
2.8801817893981934
2.862436056137085
2.8459935188293457
2.8306961059570312
2.8164069652557373
2.803009271621704
2.790404796600342
2.7785086631774902
2.7672488689422607
2.756564140319824
2.7464025020599365
2.7367186546325684
2.7274742126464844
2.7186341285705566
2.710169792175293
2.702054262161255
2.6942644119262695
2.686779260635376
2.6795802116394043
2.67264986038208
2.6659741401672363
2.6595375537872314
2.6533281803131104
2.647334098815918
2.6415445804595947
2.6359493732452393
2.6305394172668457
2.6253063678741455
2.620241165161133
2.615337610244751
2.6105875968933105
2.605985403060913
2.6015243530273438
2.597198724746704
2.5930025577545166

In [8]:
for i in range(5):
    out = []
    ix1 = 0
    ix2 = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix1, ix2]), num_classes=27).float()           
        xenc = torch.cat((xenc[0],xenc[1]), dim = 0)
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(0, keepdims=True)

        ix1 = ix2
        ix2 = torch.multinomial(p, num_samples=1, replacement=True).item()

        out.append(itos[ix2])
        if ix2 == 0:
            break

    print(''.join(out))

nelo.
javtloshayono.
carine.
gr.
brqeeldia.


The loss is better than the bigram model, and the names also look more sensible, in the limited amount I generated.

E02: split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?

In [9]:
from torch.utils.data import random_split

words = list(words)
train, dev, test = random_split(words, [0.8, 0.1, 0.1], generator=g)
train = list(train)
dev = list(dev)
test = list(test)

In [10]:
def create_bigram_dataset(words):
    X, y = [], []
    for w in words:
        chs = ['.'] + list(w) + ['.']
        for ch1, ch2 in zip(chs, chs[1:]):
            ix1 = stoi[ch1]
            ix2 = stoi[ch2]
            X.append(ix1)
            y.append(ix2)
    X = torch.tensor(X)
    num = X.nelement()
    y = torch.tensor(y)
    return X, y, num

class Bigram:
    def __init__(self, X, y, num, L2_weight = 0.01, lr = 50) -> None:
        self.X = X #assume already encoded
        self.y = y
        self.W = torch.randn((27, 27), requires_grad=True, generator=g)
        self.L2_weight = L2_weight
        self.learn_rate = lr
        self.num = num

    def train(self, num_loops=100, ce=False):
        for k in range(num_loops):
            logits = self.W[self.X]
            counts = logits.exp()
            probs = counts / counts.sum(dim=1, keepdim=True)
            if not ce:
                loss = -probs[torch.arange(self.num), self.y].log().mean() + self.L2_weight * (self.W**2).mean()
            else:
                loss = F.cross_entropy(logits, self.y) + self.L2_weight * (self.W**2).mean()
            if k % 10 == 0:
                print(k, ": ", loss.item())

            self.W.grad = None
            loss.backward()

            self.W.data -= self.learn_rate * self.W.grad

    def evaluate(self, eval_set_X, eval_set_y, eval_set_num, ce=False):
        logits = self.W[eval_set_X]
        counts = logits.exp()
        probs = counts / counts.sum(dim=1, keepdim=True)
        if not ce:
            loss = -probs[torch.arange(eval_set_num), eval_set_y].log().mean()
        else:
            loss = F.cross_entropy(logits, eval_set_y)
        return loss.item()
        

In [11]:
train_X_bigram, train_y_bigram, train_num = create_bigram_dataset(train)
bigram = Bigram(X=train_X_bigram, y=train_y_bigram, num=train_num)
bigram.train(num_loops=150)

0 :  3.596994161605835
10 :  2.683786392211914
20 :  2.5783467292785645
30 :  2.5398449897766113
40 :  2.521315813064575
50 :  2.5106663703918457
60 :  2.5038278102874756
70 :  2.4991207122802734
80 :  2.4957234859466553
90 :  2.4931881427764893
100 :  2.491248607635498
110 :  2.489734649658203
120 :  2.488532304763794
130 :  2.4875645637512207
140 :  2.4867751598358154


In [12]:
dev_X_bigram, dev_y_bigram, dev_num = create_bigram_dataset(dev)
test_X_bigram, test_y_bigram, test_num = create_bigram_dataset(test)
print("dev loss: ", bigram.evaluate(eval_set_X=dev_X_bigram, eval_set_y=dev_y_bigram, eval_set_num=dev_num))
print("test loss: ", bigram.evaluate(eval_set_X=test_X_bigram, eval_set_y=test_y_bigram, eval_set_num=test_num))

dev loss:  2.4598491191864014
test loss:  2.4676754474639893


In [13]:
def create_trigram_dataset(words):
    X, y = [], []
    for w in words:
        chs = ['.', '.'] + list(w) + ['.']
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ix1 = stoi[ch1]
            ix2 = stoi[ch2]
            ix3 = stoi[ch3]
            X.append((ix1, ix2))
            y.append(ix3)
    X = torch.tensor(X)
    num = X[:, 0].nelement()
    y = torch.tensor(y)
    return X, y, num

class Trigram:
    def __init__(self, X, y, num, L2_weight = 0.01, lr = 10, seed = 0) -> None:
        self.X = X #assume already encoded
        self.y = y
        self.seed = seed if seed != 0 else torch.randint(low=1, high=50000, size=()).item()
        g = torch.Generator().manual_seed(self.seed)
        self.W = torch.randn((54, 27), requires_grad=True, generator=g)
        self.L2_weight = L2_weight
        self.learn_rate = lr
        self.num = num

    def train(self, num_loops=100, should_print=False, ce = False):
        for k in range(num_loops):
            logits = self.W[self.X[:, 0]] + self.W[27 + self.X[:, 1]]
            counts = logits.exp()
            probs = counts / counts.sum(dim=1, keepdim=True)
            if not ce:
                loss = -probs[torch.arange(self.num), self.y].log().mean() + self.L2_weight * (self.W**2).mean()
            else:
                loss = F.cross_entropy(logits, self.y) + self.L2_weight * (self.W**2).mean()

            if should_print:
                if k % 10 == 0:
                    print(k, ": ", loss.item())

            self.W.grad = None
            loss.backward()

            self.W.data -= self.learn_rate * self.W.grad

        print("train loss: ", (loss - self.L2_weight * (self.W**2).mean()).item())

    def evaluate(self, eval_set_X, eval_set_y, eval_set_num, ce=False):
        logits = self.W[eval_set_X[:, 0]] + self.W[27 + eval_set_X[:, 1]]
        counts = logits.exp()
        probs = counts / counts.sum(dim=1, keepdim=True)
        if not ce:
            loss = -probs[torch.arange(eval_set_num), eval_set_y].log().mean()
        else:
            loss = F.cross_entropy(logits, eval_set_y)
        return loss.item()
        

In [14]:
train_X_trigram, train_y_trigram, train_num_trigram = create_trigram_dataset(train)
trigram = Trigram(X=train_X_trigram, y=train_y_trigram, num=train_num_trigram)
trigram.train(num_loops=200, should_print=True)

0 :  4.221391201019287
10 :  3.2287650108337402
20 :  2.907489061355591
30 :  2.753265619277954
40 :  2.663391590118408
50 :  2.605278968811035
60 :  2.564749240875244
70 :  2.5349254608154297
80 :  2.5120980739593506
90 :  2.494081974029541
100 :  2.4795126914978027
110 :  2.4674973487854004
120 :  2.4574267864227295
130 :  2.4488685131073
140 :  2.4415085315704346
150 :  2.4351134300231934
160 :  2.4295060634613037
170 :  2.4245493412017822
180 :  2.4201390743255615
190 :  2.4161899089813232
train loss:  2.4053735733032227


In [15]:
dev_X_trigram, dev_y_trigram, dev_num_trigram = create_trigram_dataset(dev)
test_X_trigram, test_y_trigram, test_num_trigram = create_trigram_dataset(test)
print("dev loss: ", trigram.evaluate(eval_set_X=dev_X_trigram, eval_set_y=dev_y_trigram, eval_set_num=dev_num_trigram))
print("test loss: ", trigram.evaluate(eval_set_X=test_X_trigram, eval_set_y=test_y_trigram, eval_set_num=test_num_trigram))

dev loss:  2.3946874141693115
test loss:  2.405393362045288


The train, dev, and test loss for both models are all about equal.

E03: use the dev set to tune the strength of smoothing (or regularization) for the trigram model - i.e. try many possibilities and see which one works best based on the dev set loss. What patterns can you see in the train and dev set loss as you tune this strength? Take the best setting of the smoothing and evaluate on the test set once and at the end. How good of a loss do you achieve?

In [16]:
weights = [0, 0.005, 0.02, 0.1, 0.5, 1, 2, 5, 10]
for weight in weights:
    trigram_l2 = Trigram(X=train_X_trigram, y=train_y_trigram, num=train_num_trigram, L2_weight=weight, seed=42)
    trigram_l2.train(num_loops=200)
    dev_loss = trigram_l2.evaluate(eval_set_X=dev_X_trigram, eval_set_y=dev_y_trigram, eval_set_num=dev_num_trigram)
    print("L2 weight: ", weight," | dev loss: ", dev_loss)

train loss:  2.4052250385284424
L2 weight:  0  | dev loss:  2.3952605724334717
train loss:  2.404989242553711
L2 weight:  0.005  | dev loss:  2.395045518875122
train loss:  2.4044294357299805
L2 weight:  0.02  | dev loss:  2.394542694091797
train loss:  2.404585838317871
L2 weight:  0.1  | dev loss:  2.394914150238037
train loss:  2.4396750926971436
L2 weight:  0.5  | dev loss:  2.4305403232574463
train loss:  2.49212646484375
L2 weight:  1  | dev loss:  2.4834342002868652
train loss:  2.5737059116363525
L2 weight:  2  | dev loss:  2.565753698348999
train loss:  2.725024461746216
L2 weight:  5  | dev loss:  2.7189059257507324
train loss:  2.8639299869537354
L2 weight:  10  | dev loss:  2.8594419956207275


In [19]:
trigram_test_L2_tuned = Trigram(X=train_X_trigram, y=train_y_trigram, num=train_num_trigram, L2_weight=0.02, seed=42)
trigram_test_L2_tuned.train(num_loops=200)
test_loss = trigram_test_L2_tuned.evaluate(eval_set_X=test_X_trigram, eval_set_y=test_y_trigram, eval_set_num=test_num_trigram)
print(test_loss)

train loss:  2.4044294357299805
2.4063329696655273


The loss decreases as regularization weight increases at first, then increases as the regularization coefficient becomes stronger beyond 0.02. The test loss is slightly worse than the dev loss at the best parameters. This is also true before tuning the l2 weight, so it doesn't seem to make much difference in this case.

E04: we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?

I did this one in-place above replacing the encoding.

E05: look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead?

In [18]:
trigram_cross_entropy = Trigram(train_X_trigram, train_y_trigram, train_num_trigram, 0.1, 10, 42)
trigram_cross_entropy.train(num_loops=200, ce=True)
print("dev loss: ", trigram_cross_entropy.evaluate(eval_set_X=dev_X_trigram, eval_set_y=dev_y_trigram, eval_set_num=dev_num_trigram, ce=True))
print("test loss: ", trigram_cross_entropy.evaluate(eval_set_X=test_X_trigram, eval_set_y=test_y_trigram, eval_set_num=test_num_trigram, ce=True))

train loss:  2.40458607673645
dev loss:  2.394913673400879
test loss:  2.405634880065918


In this case it looks like negative log likelihood is actually the same as cross entropy. I could see using cross entropy would be used since it's simpler and more efficient, but they are mathematically equivalent, so either works.